# Install

In [ ]:
# !pip uninstall -y numpy scipy -> gensim 및 BART와의 버전 충돌 이슈로 직접 버전 설정해줌
!pip install pandas==1.5.3 numpy==1.23.5 scipy==1.10.1 --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 42.2 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    U

In [ ]:
!pip install tensorflow==2.12.0 --force-reinstall

  Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.0 MB/s eta 0:00:00


In [ ]:
!pip install summa
!pip install transformers

In [ ]:
# 라이브러리
import pandas as pd
import numpy as np
import pickle
from urllib.request import urlretrieve, urlopen
import gzip
import zipfile
import math
import nltk
import re
from tqdm import tqdm
tqdm.pandas()  # <- 이걸 실행해야 progress_apply가 활성화됨

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from summa.summarizer import summarize

nltk.download('punkt_tab')

# 데이터 분할 및 인코더
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# 토큰화 및 패딩
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 시각화 라이브러리
import matplotlib.pyplot as plt

# 성능 평가 라이브러리
from sklearn.metrics import mean_absolute_error, mean_squared_error

print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')
print(f'tensorflow version: {tf.__version__}')       # 2.13.0 등

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


pandas version: 1.5.3
numpy version: 1.23.5
tensorflow version: 2.12.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Config/Function

In [ ]:
DATA_SAVE_PATH = '/content/drive/MyDrive/박민경_논문1/dataset/prep'
# CATEGORY = 'yelp'
# USER_ID = 'user_id'
# ITEM_ID = 'rest_id'
# REVIEW_COL = 'review_text'
# RATING_COL = 'review_stars'

CATEGORY = 'movie'
USER_ID = 'userID'
ITEM_ID = 'itemID'
RATING_COL = 'rating'
REVIEW_COL = 'reviewtext'

mean_seq_length = 44                   # book = 118, movie = 44, yelp = 113

In [ ]:
# Functions: prep, embedding, summarization with TextRank
## 1) .을 공백으로 처리했을 때 ''인지 확인하는 함수
def clean_text(text):
  cleaned = text.replace('.', '').strip()
  return cleaned == ''


# 2) 리뷰 데이터에 '.', '..' 등의 리뷰인 경우 제거하여 처리
def is_meaningful_sentence_list(sentence_list):
    if not isinstance(sentence_list, list):
        return False
    cleaned = [s.strip() for s in sentence_list if s.strip()]
    return len(cleaned) > 0


# 3) embedding
def embed_review_sentences(review_list, glove_dict, dim=300):
    sentences = [s.strip() for s in review_list if s.strip()]  # 공백 제거 및 비어 있는 문장 제외
    embeddings = []

    for sentence in sentences:
        tokens = word_tokenize(sentence)
        vectors = [glove_dict[token] for token in tokens if token in glove_dict]
        if vectors:
            embedding = np.mean(vectors, axis=0)
        else:
            embedding = np.zeros(dim)
        embeddings.append(embedding)

    return sentences, embeddings

# 4) summarize
def safe_summarize(text, ratio=0.6):
    if len(text.split(' ') ) > 250000:
      text = text[:250000]
    try:
        summary = summarize(text, ratio=ratio)
        return summary if summary and summary.strip() else text
    except Exception:
        return text  # 예외 발생 시 원문 그대로


def summarize_Textrank(df, user_item = 'user', ratio = 0.6):
    df[f'{user_item}_summary'] = df['Reviews_origin'].progress_apply(lambda x: safe_summarize(x, ratio=ratio))
    df[f'{user_item}_summary'] = df[f'{user_item}_summary'].apply(lambda x: x.replace('.\n', '. '))

    return df

In [ ]:
# 텍스트 패딩을 생성한 후, 리스트의 형태로 DataFrame에 저장
def filter_sequences(sequences, max_words):
    return [[idx for idx in seq if idx < max_words] for seq in sequences]


# 6) 텍스트 패딩을 생성한 후, 리스트의 형태로 DataFrame에 저장
def create_pad_sequence(df, col_nm, max_seq_len, max_words):
    ## 토큰화 및 패딩
    text_sequence = filter_sequences(tokenizer.texts_to_sequences(df[col_nm]), max_words)
    # text_sequence = tokenizer.texts_to_sequences(df[col_nm])
    padding_result = pad_sequences(text_sequence, maxlen=max_seq_len, padding='post')
    padding_result # 새로 만들어준 사전에서의 인덱스

    ## 'padded_sequences' 열을 df에 추가하고, 패딩이 적용된 시퀀스의 내용을 할당
    df[col_nm + '_padded_sequences'] = list(padding_result)

    ## 'padded_sequences' 열의 데이터 타입을 object로 변경하여 각 요소를 리스트 형태로 저장
    df[col_nm + '_padded_sequences'] = df[col_nm + '_padded_sequences'].apply(lambda x: list(x))



def merge_textrank_bart(path, which_user_item, id_col):
    # load/rename textrank
    df_textrank = pd.read_pickle(path + which_user_item+'_textrank.pkl')

    df_textrank = df_textrank.rename(columns = {
        id_col:which_user_item,
        f'{which_user_item}_summary_padded_sequences':f'{which_user_item}_embedding_textrank'})

    # load bart
    df_bart = pd.read_pickle(path + which_user_item+'_bart.pkl')

    ## prep id column
    if which_user_item+'_id' in df_bart.columns:
        df_bart = df_bart.rename(columns = {which_user_item+'_id':which_user_item})
    if id_col in df_bart.columns:
        df_bart = df_bart.rename(columns = {id_col:'item'})

    ## rename bart column
    df_bart = df_bart.rename(columns = {'bart_embedding':f'{which_user_item}_embedding_bart'})

    # merge
    df_merge = pd.merge(
        df_textrank[[which_user_item, f'{which_user_item}_embedding_textrank']],
        df_bart[[which_user_item, f'{which_user_item}_embedding_bart']],
        on = which_user_item)

    return df_merge

In [ ]:
# 리뷰 길이 0인 데이터 전처리 함수
def prep_review_length_zero(df, col):
    df_empty = df[df[col].apply(lambda x: True if clean_text(x) == True else False)]

    if df_empty.shape[0] > 0:
        print(f'리뷰 길이가d 0인 데이터 수: {df_empty.shape[0]}')

        df = df[df[col].apply(lambda x: True if clean_text(x) == False else False)]
        after_shape = df[df[col].apply(lambda x: True if clean_text(x) == True else False)].shape[0]

        print('-'*50, '리뷰 길이가 0인 데이터 전처리 완료', '-'*50)
        print(f'리뷰 길이가 0인 데이터 수: {after_shape}')
    else:
        print('리뷰 길이 0인 데이터 없음')
    return df



# 집합 데이터 분할 및 저장 함
def split_data_save(df, num, WHICH_USER_ITEM, DATA_SAVE_PATH = DATA_SAVE_PATH, CATEGORY = CATEGORY):
    df_grouped_user_split = np.array_split(df, num)

    for i in range(0, num):
        df_grouped_user_split[i].to_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_{WHICH_USER_ITEM}_split{i+1}.pkl')

    return print('----------------finished-----------------')

# Load

In [ ]:
# GloVe 불러오기 : 우선 300차원
## download glove
# urlretrieve("http://nlp.stanford.edu/data/glove.6B.zip", filename="glove.6B.zip") # SSL certificate issue when trying to download -> the certificate for the current download URL has expired
urlretrieve("https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip", filename="glove.6B.zip") # Use a different URL
zf = zipfile.ZipFile('glove.6B.zip')
zf.extractall()
zf.close()

glove_dict = dict()
f = open('glove.6B.300d.txt', encoding="utf8") # 300차원의 GloVe 벡터를 사용

for line in f:
    word_vector = line.split()
    word = word_vector[0]
    word_vector_arr = np.asarray(word_vector[1:], dtype='float32') # 300개의 값을 가지는 array로 변환
    glove_dict[word] = word_vector_arr
f.close()

In [ ]:
# 전체 데이터 불러오기
df_reviews = pd.read_pickle(f"{DATA_SAVE_PATH}/{CATEGORY}_data.pkl")
df_reviews.head(5)

,review_id,review_useful,review_text,review_stars,rest_aver_stars,state,user_review_count,user_elite,user_friends,review_date,user_id,rest_id
0,KU_O5udG6zpxOg-VcAEodg,0,if you decide to eat here just be aware it is ...,3.0,3.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-07-07 22:09:11,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw
2,78MWkzX8uQz0kDnhlhwAAg,2,so glad we stumbled upon this restaurant it is...,4.0,4.5,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-10-29 21:54:59,mh_-eMZ6K5RLWhZyISBhwA,mOk3D7VczrDapNuUgLxUQw
3,krpCZHUj222Ha7AffGUZHQ,3,i was looking forward to a romantic dinner her...,2.0,4.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-02-11 03:07:30,mh_-eMZ6K5RLWhZyISBhwA,L4kfcADLCU4T33i7Z0CkuA
4,NFiy4sFGVAn0A9Z22WkmZA,0,not a bad dunkin donuts but certainly not the ...,3.0,1.5,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2017-03-02 22:03:57,mh_-eMZ6K5RLWhZyISBhwA,tVfJPW14AeuAHDJeleWWdQ
5,qD25Gdcd1AdfBzR3Ibx3bg,3,humptys dumplings is one of the highlights of ...,5.0,4.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2017-01-08 20:45:48,mh_-eMZ6K5RLWhZyISBhwA,gpYBhnTk4KzvvH83TsZiQg


# GROUPBY

In [ ]:
# USER: 리뷰 집합 데이터 생성
from bs4 import BeautifulSoup

grouped_user_reviews = df_reviews[[USER_ID, REVIEW_COL]].groupby(USER_ID)[REVIEW_COL].sum()
grouped_user_reviews = pd.DataFrame(grouped_user_reviews)
grouped_user_reviews.rename(columns = {REVIEW_COL:"userReviews"}, inplace = True)
grouped_user_reviews['userReviews'] = grouped_user_reviews['userReviews'].apply(
    lambda txt: BeautifulSoup(txt, "html.parser").get_text()
)

# grouped_user_reviews['userReviews'] = grouped_user_reviews['userReviews'].apply(
#     lambda txt: re.sub(r"\.", " ", txt).replace('\n', ' ')
# )


# 인덱스를 열로 변환
grouped_user_reviews.reset_index(inplace=True)

# df 생성
group_user = pd.DataFrame({
    USER_ID: grouped_user_reviews[USER_ID].values,
    "userReviews": grouped_user_reviews["userReviews"].values})

print(group_user.shape)

(102308, 2)


In [ ]:
# ITEM: 리뷰 집합 데이터 생성
grouped_item_reviews = df_reviews[[ITEM_ID, REVIEW_COL]].groupby(ITEM_ID)[REVIEW_COL].sum()
grouped_item_reviews = pd.DataFrame(grouped_item_reviews)
grouped_item_reviews.rename(columns = {REVIEW_COL:"itemReviews"}, inplace = True)
grouped_item_reviews['itemReviews'] = grouped_item_reviews['itemReviews'].apply(
    lambda txt: BeautifulSoup(txt, "html.parser").get_text()
)

# grouped_item_reviews['itemReviews'] = grouped_item_reviews['itemReviews'].apply(
#     lambda txt: re.sub(r"\.", " ", txt).replace('\n', ' ')
# )


# 인덱스를 열로 변환
grouped_item_reviews.reset_index(inplace=True)

# df 생성
group_item = pd.DataFrame({
    ITEM_ID: grouped_item_reviews[ITEM_ID].values,
    "itemReviews": grouped_item_reviews["itemReviews"].values})

print(group_item.shape)

(19622, 2)


## Prep

In [ ]:
group_user = prep_review_length_zero(group_user, 'userReviews')
group_item = prep_review_length_zero(group_item, 'itemReviews')

In [ ]:
# review set 전처리해서 사용해서 원본 review set column 이름 변경
group_user = group_user.rename(columns = {'userReviews':'Reviews_origin'})
group_item = group_item.rename(columns = {'itemReviews':'Reviews_origin'})

# 데이터프레임을 지정된 경로에 저장
group_user.to_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_user.pkl', protocol=4)

# 데이터프레임을 지정된 경로에 저장
group_item.to_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_item.pkl', protocol=4)

In [ ]:
# 데이터프레임을 지정된 경로에서 불러오기
group_user = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_user.pkl')
group_item = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_item.pkl')

print(group_user.shape)
print(group_item.shape)

group_item.head(5)

In [ ]:
split_data_save(group_user, 5, 'user')
split_data_save(group_item, 5, 'item')

# Summarize

## 1)요약

1. 분할해서 요약하거나
2. 전체 유저/아이템 데이터 요약하거나

In [ ]:
# 분할 데이터 불러오기
WHICH_USER_ITEM = 'user'
WHICH_VERSION = '4'

if WHICH_VERSION == None:
  df_split = pd.read_pickle(f'{DATA_SAVE_PATH}/textrank_mid_output/{CATEGORY}_{WHICH_USER_ITEM}_{WHICH_USER_ITEM}.pkl')
else:
  df_split = pd.read_pickle(f'{DATA_SAVE_PATH}/textrank_mid_output/{CATEGORY}_{WHICH_USER_ITEM}_split{WHICH_VERSION}.pkl')

print(df_split.shape)
df_split.head(5)

(13702, 2)


,userID,Reviews_origin
41142,AGGYU65HTLHSS5E4MNE5MKJJYQFQ,interestinggreat topics discussedgreat topicgr...
41143,AGGYVACW6LCHZEHXWCIAD2VREKFA,this is one of those when i heard someone had ...
41144,AGGYZO4BMNQ4BNTNOF46XMNFVOEQ,very different but a good moviereally like thi...
41145,AGGZ6RREXQM6AFPI2DM3X4Y2ORQA,glad that amazon decided to make a series of t...
41146,AGGZAEL4ZYEWDYO2GK6FD7GUIBXA,overall a horrible movie story line acting eve...


### a.batch summarization

In [ ]:
# 배치로 요약 돌리기
lst_df = []
total = len(df_split)
batch_size = 100
ratio = 0.6

for start in tqdm(range(0, total, batch_size)):
    end = min(start + batch_size, total)
    print(f'\nfrom {start} to {end}: summarizing\n')
    _df_batch = df_split.iloc[start:end].copy()
    _df_batch['item_summary'] = _df_batch['Reviews_origin'].apply(lambda x: safe_summarize(x, ratio=ratio))
    _df_batch['item_summary'] = _df_batch['item_summary'].apply(lambda x: x.replace('.\n', '. '))
    lst_df.append(_df_batch)

  0%|          | 0/138 [00:00<?, ?it/s]


from 0 to 100: summarizing



  1%|          | 1/138 [00:01<02:23,  1.04s/it]


from 100 to 200: summarizing



  1%|▏         | 2/138 [00:01<02:09,  1.05it/s]

...


from 13500 to 13600: summarizing



 99%|█████████▊| 136/138 [21:30<00:06,  3.22s/it]


from 13600 to 13700: summarizing



100%|██████████| 138/138 [21:30<00:00,  9.35s/it]


from 13700 to 13702: summarizing



In [ ]:
df_split = pd.concat(lst_df, ignore_index=True)

# 분할한 요약 데이터 저장
if WHICH_VERSION == None:
  _path = f'{DATA_SAVE_PATH}/textrank_mid_output/{CATEGORY}_{WHICH_USER_ITEM}_textrank.pkl'
else:
  _path = f'{DATA_SAVE_PATH}/textrank_mid_output/{CATEGORY}_{WHICH_USER_ITEM}_textrank{WHICH_VERSION}.pkl'

df_split.to_pickle(_path)
df_split = pd.read_pickle(_path)
df_split.head(10)

,userID,Reviews_origin,user_summary
0,AGGYU65HTLHSS5E4MNE5MKJJYQFQ,interestinggreat topics discussedgreat topicgr...,interestinggreat topics discussedgreat topicgr...
1,AGGYVACW6LCHZEHXWCIAD2VREKFA,this is one of those when i heard someone had ...,this is one of those when i heard someone had ...
2,AGGYZO4BMNQ4BNTNOF46XMNFVOEQ,very different but a good moviereally like thi...,very different but a good moviereally like thi...
3,AGGZ6RREXQM6AFPI2DM3X4Y2ORQA,glad that amazon decided to make a series of t...,glad that amazon decided to make a series of t...
4,AGGZAEL4ZYEWDYO2GK6FD7GUIBXA,overall a horrible movie story line acting eve...,overall a horrible movie story line acting eve...
5,AGGZEDMFZOFZA6VZM6OW3DTP72XA,cast perfection great movie songs and message....,cast perfection great movie songs and message....
6,AGGZFRDT32KS6CA5HT5FCZQBV5JQ,yes the acting had some rough cheesy spots and...,yes the acting had some rough cheesy spots and...
7,AGGZHAJMU7STUQ7SZDIJEPMHGKVA,great romantic story based on a real life stor...,great romantic story based on a real life stor...
8,AGGZHEPPW6SSRPQ57EUBDO46OMBA,amazing show star based solely on that. there...,there are slim special features. thats it.off ...
9,AGGZIJ7VSESHC4DG7T7F4D4YTXIQ,i cant believe this film was nominated for ac...,i cant believe this film was nominated for ac...


#### 👀debugging
아이템 데이터를 분할해서 요약하는데, 특정 분할 데이터에서 자꾸 세션이 다운되거나 뻑남\
-> 리뷰 길이를 단어 기준으로 계산해서 뻑이난 지점(=item_data32에서 70~80번째 사이에서 남)에서 리뷰 길이 확인함
(대조군: 전체 아이템 리뷰 데이터의 단어 기준 리뷰 길이)

In [ ]:
# 전체 아이템 데이터의 리뷰 길이(단어 기준)
group_item = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_item.pkl')
group_item = group_item.rename(columns = {'itemReviews':'Reviews_origin'})
sorted_list1 = sorted(list(group_item['Reviews_origin'].apply(lambda x: len(x.split(' '))).unique()), reverse=True)
print(sorted_list1[:30])

[317878, 245546, 232253, 212843, 206482, 198460, 189288, 186930, 166917, 165418, 158072, 157204, 146586, 141663, 140428, 137724, 133981, 131070, 128240, 128116, 128014, 127115, 124616, 120125, 117282, 116831, 116306, 115028, 114681, 113151]


In [ ]:
# 해당 데이터의 리뷰 길이(단어 기준)
sorted_list = sorted(list(df_split['Reviews_origin'].apply(lambda x: len(x.split(' '))).unique()), reverse=True)
print(sorted_list[:30])

[317878, 212843, 165418, 128240, 124616, 120125, 104893, 96450, 85470, 82966, 81659, 81470, 73239, 70291, 63084, 62787, 59054, 58638, 58180, 57695, 54776, 54690, 52042, 51112, 49124, 48122, 46893, 45961, 44890, 43868]


In [ ]:
# 해당 데이터 중 세션 다운되는 구간의 리뷰 길이 확인
df_split.iloc[70:80]['Reviews_origin'].apply(lambda x: len(x.split(' ')))

,Reviews_origin
4535,1623
4536,3881
4537,10216
4538,897
4539,1627
4540,5090
4541,9277
4542,317878
4543,954
4544,560


---------

### b.total summarization

In [ ]:
# Textrank로 요약하기
df_split = summarize_Textrank(df_split)

print(df_split.shape)
df_split.head(5)

## 2)분할 데이터 병합
- 분할해서 요약했을 때 하나로 합치기

In [ ]:
df_summary = pd.DataFrame()
LAST_SUMMARY_NUM = 5
WHICH_USER_ITEM = 'item'

for i in range(1, LAST_SUMMARY_NUM+1):
    _df_temp = pd.read_pickle(f'{DATA_SAVE_PATH}/textrank_mid_output/{CATEGORY}_{WHICH_USER_ITEM}_textrank{i}.pkl')
    df_summary = pd.concat([df_summary, _df_temp])

# user/item 리뷰 요약 집합 저장
df_summary.to_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_{WHICH_USER_ITEM}_textrank.pkl')

In [ ]:
# user/item 리뷰 요약 집합 불러오기
df_summary = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_{WHICH_USER_ITEM}_textrank.pkl')
print(f'{WHICH_USER_ITEM}:', df_summary.shape)
df_summary.head(5)

item: (46281, 3)


,itemID,Reviews_origin,item_summary
0,0767802470,what a hell it must have been to crew one of t...,great characters and excellent storyline will ...
1,0767802497,i this is a action movie of sorts if u dont li...,i this is a action movie of sorts if u dont li...
2,0767802519,i liked it because the actually story was real...,i liked it because the actually story was real...
3,0767802535,one of the best movies ever made...about two i...,one of the best movies ever made...about two i...
4,0767803434,great moviea fiction that could be written tod...,subterfuge at its finest.awesome jobi dont und...


# Embedding - Glove

## 1)User

In [ ]:
df_user_summary = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_user_textrank.pkl')
df_user_summary.head(3)

,userID,Reviews_origin,user_summary,user_summary_padded_sequences
0,AE223CNRS5AQOKRWCFPRBUNJ5T2Q,entertaininglove the animationfunnyentertainin...,entertaininglove the animationfunnyentertainin...,"[26910, 1, 235, 23, 821, 1, 34, 626, 13914, 49..."
1,AE22FB6A4Z54ZUJZMX2QGRWSNIZA,this show is so fresh i honestly feel from the...,this show is so fresh i honestly feel from the...,"[9927, 4, 1144, 1133, 210, 14, 71, 8933, 595, ..."
2,AE22HYC5EJJ4AUZ55HTCJFPKJFHQ,got my money back right away. so no big deal....,got my money back right away. the language and...,"[870, 11, 352, 9, 28, 150, 7, 3319, 1732, 6252..."


In [ ]:
# 단어 인덱스 맵 생성 및 임베딩 행렬 준비
## 토크나이저 초기화 및 학습
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df_user_summary['user_summary'])

max_words = 50000
word_index = tokenizer.word_index
total_words = min(max_words, len(word_index))
print(f'total_words: {total_words}')

total_words: 50000


In [ ]:
## 단어 인덱스 맵 생성 및 임베딩 행렬 준비: embedding dimension=300
user_embedding_matrix = np.zeros((total_words + 1, 300))

## 새로 만든 사전에 맞게 임베딩 해줘야함
for word, i in word_index.items():
    if i >= max_words:
        continue
    if word in glove_dict:
        user_embedding_matrix[i] = glove_dict[word]

print(user_embedding_matrix)
user_embedding_matrix.shape

[[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.04656     0.21318001 -0.0074364  ...  0.0090611  -0.20988999
   0.053913  ]
 [ 0.038466   -0.039792    0.082747   ... -0.33427     0.011807
   0.059703  ]
 ...
 [ 0.0053435   0.36649001  0.38389    ... -0.35992    -0.1323
  -0.53593999]
 [ 0.01577     0.56884003  0.41288999 ...  0.040484   -0.03643
  -0.36094999]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]


(50001, 300)

In [ ]:
create_pad_sequence(df_user_summary, 'user_summary', max_seq_len = mean_seq_length, max_words=total_words)
df_summary.head(10)

,userID,Reviews_origin,user_summary,user_summary_padded_sequences
0,AE223CNRS5AQOKRWCFPRBUNJ5T2Q,entertaininglove the animationfunnyentertainin...,entertaininglove the animationfunnyentertainin...,"[26910, 1, 235, 23, 821, 1, 34, 626, 13914, 49..."
1,AE22FB6A4Z54ZUJZMX2QGRWSNIZA,this show is so fresh i honestly feel from the...,this show is so fresh i honestly feel from the...,"[9927, 4, 1144, 1133, 210, 14, 71, 8933, 595, ..."
2,AE22HYC5EJJ4AUZ55HTCJFPKJFHQ,got my money back right away. so no big deal....,got my money back right away. the language and...,"[870, 11, 352, 9, 28, 150, 7, 3319, 1732, 6252..."
3,AE22IEG7RTLEV5TXRK5DCOKFZJLQ,doubting there is any accuracy to this drama i...,doubting there is any accuracy to this drama i...,"[141, 121, 5, 1, 6, 272, 198, 7, 58, 4, 331, 4..."
4,AE22KH6ZLWIZPZ3UM5SFMUKZ7UAA,i started to watch lore and loved it season w...,i started to watch lore and loved it season w...,"[168, 8, 922, 2, 686, 19, 4, 280, 83, 185, 19,..."
5,AE22KUGFMITQ24E7J7QVZEYAW7CA,for such a consistently great director this fi...,for such a consistently great director this fi...,"[9, 19, 71, 229, 4, 280, 703, 191, 1, 204, 189..."
6,AE22PFS2QZMV5EPBL7Y6TLMVRLJQ,this is a great series. sensitive fastpaced ...,this is a great series. the setting in montrea...,"[7, 3778, 1205, 337, 882, 1, 213, 4, 123, 1454..."
7,AE22QDEN6KCC2YYSJWUIT6ARADFQ,good movie has depth.while the fee for rentin...,good movie has depth.while the fee for rentin...,"[5, 1, 100, 55, 1337, 112, 109, 20, 226, 140, ..."
8,AE22QSIR4YRJG5V5NHOL7KDAN3ZQ,well thought out sentences might be too much f...,well thought out sentences might be too much f...,"[3, 76, 105, 210, 102, 436, 219, 148, 11, 6168..."
9,AE22RGXR4645FMVQZJUYIOM5NTMQ,cant wait this k set. im glad v and vi are om...,loved seeing bale in another comic book movie....,"[1064, 21, 1, 385, 443, 29, 35, 1, 330, 64, 10..."


In [ ]:
np.max(list(df_summary['user_summary_padded_sequences'].values))

49996

In [ ]:
# save as npy
np.save(f'{DATA_SAVE_PATH}/{CATEGORY}_user_glove_dic.npy', user_embedding_matrix)

# user 리뷰 요약 시퀀스 저장
df_user_summary.to_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_user_textrank.pkl')

In [ ]:
# 저장한 데이터 확인
df_user_textrank = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_user_textrank.pkl')

print(np.max(list(df_user_textrank['user_summary_padded_sequences'].values)))
df_user_textrank.head(5)

49996


,userID,Reviews_origin,user_summary,user_summary_padded_sequences
0,AE223CNRS5AQOKRWCFPRBUNJ5T2Q,entertaininglove the animationfunnyentertainin...,entertaininglove the animationfunnyentertainin...,"[26910, 1, 235, 23, 821, 1, 34, 626, 13914, 49..."
1,AE22FB6A4Z54ZUJZMX2QGRWSNIZA,this show is so fresh i honestly feel from the...,this show is so fresh i honestly feel from the...,"[9927, 4, 1144, 1133, 210, 14, 71, 8933, 595, ..."
2,AE22HYC5EJJ4AUZ55HTCJFPKJFHQ,got my money back right away. so no big deal....,got my money back right away. the language and...,"[870, 11, 352, 9, 28, 150, 7, 3319, 1732, 6252..."
3,AE22IEG7RTLEV5TXRK5DCOKFZJLQ,doubting there is any accuracy to this drama i...,doubting there is any accuracy to this drama i...,"[141, 121, 5, 1, 6, 272, 198, 7, 58, 4, 331, 4..."
4,AE22KH6ZLWIZPZ3UM5SFMUKZ7UAA,i started to watch lore and loved it season w...,i started to watch lore and loved it season w...,"[168, 8, 922, 2, 686, 19, 4, 280, 83, 185, 19,..."


## 2)Item

In [ ]:
df_item_summary = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_item_textrank.pkl')
df_item_summary.head(3)

,itemID,Reviews_origin,item_summary
0,0767802470,what a hell it must have been to crew one of t...,great characters and excellent storyline will ...
1,0767802497,i this is a action movie of sorts if u dont li...,i this is a action movie of sorts if u dont li...
2,0767802519,i liked it because the actually story was real...,i liked it because the actually story was real...


In [ ]:
# 2. Item review
## 토크나이저 초기화 및 학습
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df_item_summary['item_summary'])

max_words = 50000
word_index = tokenizer.word_index
total_words = min(max_words, len(word_index))
print(f'total_words: {total_words}')

total_words: 50000


In [ ]:
## 단어 인덱스 맵 생성 및 임베딩 행렬 준비: embedding dimension=300
item_embedding_matrix = np.zeros((total_words + 1, 300))

## 새로 만든 사전에 맞게 임베딩 해줘야함
for word, i in word_index.items():
    if i >= max_words:
        continue
    if word in glove_dict:
        item_embedding_matrix[i] = glove_dict[word]

print(item_embedding_matrix)
item_embedding_matrix.shape

[[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.04656     0.21318001 -0.0074364  ...  0.0090611  -0.20988999
   0.053913  ]
 [ 0.038466   -0.039792    0.082747   ... -0.33427     0.011807
   0.059703  ]
 ...
 [-0.59227002 -0.69761997  0.14       ...  0.56614    -0.39524999
   0.15745001]
 [ 0.12574001  0.37318     0.17019001 ... -0.53294998  0.043552
  -0.32440001]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]


(50001, 300)

In [ ]:
## max_sequ_len = 데이터별 평균 시퀀스 길이로 두고 embedding 생성
create_pad_sequence(df_item_summary, 'item_summary', max_seq_len = mean_seq_length, max_words=total_words)
df_item_summary.head(10)

,itemID,Reviews_origin,item_summary,item_summary_padded_sequences
0,0767802470,what a hell it must have been to crew one of t...,great characters and excellent storyline will ...,"[57, 5, 21013, 51, 19797, 1036, 11759, 2, 5263..."
1,0767802497,i this is a action movie of sorts if u dont li...,i this is a action movie of sorts if u dont li...,"[10, 9, 1487, 65, 18, 62, 18, 79, 31, 493, 2, ..."
2,0767802519,i liked it because the actually story was real...,i liked it because the actually story was real...,"[51, 61, 134, 2, 106, 397, 147, 1, 2822, 5, 20..."
3,0767802535,one of the best movies ever made...about two i...,one of the best movies ever made...about two i...,"[378, 5, 2140, 2338, 8, 671, 66, 1, 12, 1888, ..."
4,0767803434,great moviea fiction that could be written tod...,subterfuge at its finest.awesome jobi dont und...,"[41, 7, 12, 21, 1588, 2, 302, 9, 21, 192, 42, ..."
5,076780421X,greatdefinitely worth gettingit was a good mov...,greatdefinitely worth gettingit was a good mov...,"[655, 10, 173, 24, 67, 26138, 42, 7, 295, 1, 1..."
6,0767805763,greatloved itbizarre. but good acting. love v...,love voighti love this movie its one of the be...,"[42, 7, 12, 29, 30, 5, 1, 107, 2339, 6104, 53,..."
7,0767806239,good moviethis is a movie that is timeless a m...,good moviethis is a movie that is timeless a m...,"[22, 969, 8, 3, 12, 14, 8, 2424, 3, 12, 14, 24..."
8,076780676X,it was fun to watch. mostly predictable. somet...,it was fun to watch. something that is safe fo...,"[3, 2408, 1907, 44, 17268, 133, 192, 954, 24, ..."
9,0767806808,funny moviereally funny movie in my opinion. i...,funny moviereally funny movie in my opinion. i...,"[15, 1, 116, 198, 6, 244, 7, 12, 120, 2185, 1,..."


In [ ]:
np.max(list(df_item_summary['item_summary_padded_sequences'].values))

49997

In [ ]:
# save as npy
np.save(f'{DATA_SAVE_PATH}/{CATEGORY}_item_glove_dic.npy', item_embedding_matrix)

# item 리뷰 요약 시퀀스 저장
df_item_summary.to_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_item_textrank.pkl')

In [ ]:
# 저장한 데이터 확인
df_item_textrank = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_item_textrank.pkl')

print(np.max(list(df_item_textrank['item_summary_padded_sequences'].values)))
df_item_summary.head(5)

49997


,itemID,Reviews_origin,item_summary,item_summary_padded_sequences
0,0767802470,what a hell it must have been to crew one of t...,great characters and excellent storyline will ...,"[57, 5, 21013, 51, 19797, 1036, 11759, 2, 5263..."
1,0767802497,i this is a action movie of sorts if u dont li...,i this is a action movie of sorts if u dont li...,"[10, 9, 1487, 65, 18, 62, 18, 79, 31, 493, 2, ..."
2,0767802519,i liked it because the actually story was real...,i liked it because the actually story was real...,"[51, 61, 134, 2, 106, 397, 147, 1, 2822, 5, 20..."
3,0767802535,one of the best movies ever made...about two i...,one of the best movies ever made...about two i...,"[378, 5, 2140, 2338, 8, 671, 66, 1, 12, 1888, ..."
4,0767803434,great moviea fiction that could be written tod...,subterfuge at its finest.awesome jobi dont und...,"[41, 7, 12, 21, 1588, 2, 302, 9, 21, 192, 42, ..."


# Merge

In [ ]:
df_user_tr_bart = merge_textrank_bart(
    path = f"{DATA_SAVE_PATH}/{CATEGORY}_", which_user_item = 'user', id_col = USER_ID)
df_user_tr_bart.head(5)

,user,user_embedding_textrank,user_embedding_bart
0,AE223CNRS5AQOKRWCFPRBUNJ5T2Q,"[26910, 1, 235, 23, 821, 1, 34, 626, 13914, 49...","[-0.024359785, -0.8241518, 0.030495215, -1.543..."
1,AE22FB6A4Z54ZUJZMX2QGRWSNIZA,"[9927, 4, 1144, 1133, 210, 14, 71, 8933, 595, ...","[0.48813066, 0.3011386, 2.4315197, -3.2398725,..."
2,AE22HYC5EJJ4AUZ55HTCJFPKJFHQ,"[870, 11, 352, 9, 28, 150, 7, 3319, 1732, 6252...","[0.55364466, -0.7212269, -0.8160649, -1.709610..."
3,AE22IEG7RTLEV5TXRK5DCOKFZJLQ,"[141, 121, 5, 1, 6, 272, 198, 7, 58, 4, 331, 4...","[0.08958933, 0.31884375, -1.5628182, -2.429583..."
4,AE22KH6ZLWIZPZ3UM5SFMUKZ7UAA,"[168, 8, 922, 2, 686, 19, 4, 280, 83, 185, 19,...","[-0.40517938, 0.15798128, -0.23898572, -0.7440..."


In [ ]:
df_item_tr_bart = merge_textrank_bart(
    path = f"{DATA_SAVE_PATH}/{CATEGORY}_", which_user_item = 'item', id_col = ITEM_ID)
df_item_tr_bart.head(5)

,item,item_embedding_textrank,item_embedding_bart
0,0767802470,"[57, 5, 21013, 51, 19797, 1036, 11759, 2, 5263...","[-0.45721912, 0.22749305, 0.76964325, -0.61435..."
1,0767802497,"[10, 9, 1487, 65, 18, 62, 18, 79, 31, 493, 2, ...","[0.31335288, 0.03406208, 0.58331335, -2.655624..."
2,0767802519,"[51, 61, 134, 2, 106, 397, 147, 1, 2822, 5, 20...","[-0.5766005, -0.5311635, 1.1096023, -2.885747,..."
3,0767802535,"[378, 5, 2140, 2338, 8, 671, 66, 1, 12, 1888, ...","[-0.28083846, -0.22997823, -0.5919732, -2.8058..."
4,0767803434,"[41, 7, 12, 21, 1588, 2, 302, 9, 21, 192, 42, ...","[-0.55130845, -0.5349555, 0.3148671, -2.708737..."


# Mapping

In [ ]:
# load rating, userID, itemID
df_all = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_data.pkl')
df_all = df_all.rename(columns = {USER_ID:'user', ITEM_ID:'item', RATING_COL:'rating'})
df_all.head(5)

,rating,item,user,reviewtext,date,year
0,1.0,B08X6JFYT9,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,so for all of those who think by buying the cu...,2021-03-01 15:57:45.651000064,2021
1,5.0,B07PND31HD,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,i wish there were more delightful movies like ...,2020-10-22 17:18:42.180999936,2020
2,1.0,B08DX1WFWK,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,spoilers this was just so ridiculous. so we ge...,2020-10-14 06:42:45.477999872,2020
3,5.0,B005LTHT72,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,i dont think there is a movie that makes me la...,2020-10-11 05:55:28.479000064,2020
4,2.0,B07R6YJMW5,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,i wanted to like this movie but the acting was...,2020-10-06 00:44:30.933000192,2020


In [ ]:
# merge user/item data
df_final_output = pd.merge(df_all[['user', 'item', 'rating']].copy(), df_user_tr_bart, on = 'user')
df_final_output = pd.merge(df_final_output, df_item_tr_bart, on = 'item')

df_final_output.head(5)

,user,item,rating,user_embedding_textrank,user_embedding_bart,item_embedding_textrank,item_embedding_bart
0,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,B08X6JFYT9,1.0,"[1, 34, 8, 14, 19, 160, 423, 104, 123, 19, 160...","[0.93794405, -0.08286553, -0.043052204, -3.257...","[27, 35, 24, 71, 8, 431, 202, 65, 4, 31, 30, 3...","[-0.16816601, -0.38978252, 0.44575274, -3.7842..."
1,AFKHQSGVE53BKPT4DQFJGRFMCLWQ,B08X6JFYT9,5.0,"[81, 26747, 31221, 3226, 1013, 2, 15479, 579, ...","[0.20747374, -0.6719453, 0.25165808, -3.481782...","[27, 35, 24, 71, 8, 431, 202, 65, 4, 31, 30, 3...","[-0.16816601, -0.38978252, 0.44575274, -3.7842..."
2,AH4HCQLXK72IFJESRHIP5MOGJG4Q,B08X6JFYT9,1.0,"[75, 2861, 580, 6, 2019, 8, 697, 64, 609, 53, ...","[-1.2395746, 1.521351, 1.3411634, -1.8987955, ...","[27, 35, 24, 71, 8, 431, 202, 65, 4, 31, 30, 3...","[-0.16816601, -0.38978252, 0.44575274, -3.7842..."
3,AEK3GBHW4JC2YMMVNDMUIXTONH6Q,B08X6JFYT9,5.0,"[7, 76, 155, 20, 408, 65, 231, 44, 9, 6, 35, 1...","[-0.18107454, -0.7605884, -1.2539482, -1.11232...","[27, 35, 24, 71, 8, 431, 202, 65, 4, 31, 30, 3...","[-0.16816601, -0.38978252, 0.44575274, -3.7842..."
4,AHRPCIOV6TWLFMNASTQLLTKBUIEQ,B08X6JFYT9,1.0,"[3919, 27, 1, 47, 10, 10, 6, 92, 24, 1, 674, 5...","[0.66322744, 0.3501363, 0.8260165, -1.8900322,...","[27, 35, 24, 71, 8, 431, 202, 65, 4, 31, 30, 3...","[-0.16816601, -0.38978252, 0.44575274, -3.7842..."


# Save File

In [ ]:
# textrank 시퀀스 생성 후 저장
df_final_output.to_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_raw_embedding.pkl')
df_final_output = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_raw_embedding.pkl')

print(np.max(list(df_final_output['user_embedding_textrank'].values)))
print(np.max(list(df_final_output['item_embedding_textrank'].values)))
print(df_final_output.shape)
df_final_output.head(3)

49996
49997
(731248, 7)


,user,item,rating,user_embedding_textrank,user_embedding_bart,item_embedding_textrank,item_embedding_bart
0,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,B08X6JFYT9,1.0,"[1, 34, 8, 14, 19, 160, 423, 104, 123, 19, 160...","[0.93794405, -0.08286553, -0.043052204, -3.257...","[27, 35, 24, 71, 8, 431, 202, 65, 4, 31, 30, 3...","[-0.16816601, -0.38978252, 0.44575274, -3.7842..."
1,AFKHQSGVE53BKPT4DQFJGRFMCLWQ,B08X6JFYT9,5.0,"[81, 26747, 31221, 3226, 1013, 2, 15479, 579, ...","[0.20747374, -0.6719453, 0.25165808, -3.481782...","[27, 35, 24, 71, 8, 431, 202, 65, 4, 31, 30, 3...","[-0.16816601, -0.38978252, 0.44575274, -3.7842..."
2,AH4HCQLXK72IFJESRHIP5MOGJG4Q,B08X6JFYT9,1.0,"[75, 2861, 580, 6, 2019, 8, 697, 64, 609, 53, ...","[-1.2395746, 1.521351, 1.3411634, -1.8987955, ...","[27, 35, 24, 71, 8, 431, 202, 65, 4, 31, 30, 3...","[-0.16816601, -0.38978252, 0.44575274, -3.7842..."
